# Zadanie 2.2: scenariusze z tematu i dwa modele dawki

Cztery plany napromieniania tego samego guza (plateau o promieniu 6, gęstość $K = 1$):

| Plan | Pole wiązki | $\beta$ |
| --- | --- | --- |
| pole pokrywa guz, duża dawka | $s = 0$, $r = 20$ | 1,5 |
| pole pokrywa guz, mała dawka | $s = 0$, $r = 20$ | 0,45 |
| pole za wąskie | $s = 0$, $r = 3$: oba brzegi guza poza polem | 1,5 |
| pole przesunięte | $s = 6$, $r = 8$: lewy brzeg guza poza polem | 1,5 |

Harmonogram: 10 frakcji co 0,4 od $t = 1$. Dawki są ilustracyjne: $\beta = 1{,}5$ zostawia po frakcji $e^{-1{,}5} \approx 22\%$
komórek, klinicznie (2 Gy) to raczej $\beta \approx 0{,}12$ (issue #10).

Dwa modele dawki: **impuls** (człon $-\beta R(x) U(t) u$, model bazowy) i **skok LQ** ($u \leftarrow u\,e^{-\beta R(x)}$ w chwili frakcji,
jak w TumorTwin). Liczymy solverem B (`cn`), $\Delta x = 0{,}1$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from offtarget.config import FIGURES, TABLES, seed_everything
from offtarget.model import Params, Schedule, beam, grid
from offtarget.observables import front_positions, min_mass, time_to_regrowth, tumor_mass
from offtarget.solvers import simulate

rng = seed_everything()

x, dx = grid(80.0, 800)
t = np.round(np.linspace(0.0, 10.0, 201), 10)
schedule = Schedule.regular(n=10, start=1.0, interval=0.4)
end_of_treatment = schedule.times[-1] + 3 * schedule.width

plans = {
    "pole pokrywa guz, duża dawka": Params(beta=1.5, s=0.0, r=20.0),
    "pole pokrywa guz, mała dawka": Params(beta=0.45, s=0.0, r=20.0),
    "pole za wąskie": Params(beta=1.5, s=0.0, r=3.0),
    "pole przesunięte": Params(beta=1.5, s=6.0, r=8.0),
}
runs = {(name, dose): simulate(p, schedule, x, t, method="cn", dose=dose)
        for name, p in plans.items() for dose in ("pulse", "jump")}

## Obserwable

Masa guza $M(t) = \int u\,dx$, najmniejsza masa, czas do odrostu (pierwsza chwila po minimum, w której $M$ wraca do $M(0)$),
brzegi guza w $t = 10$ (poziom $u = 0{,}5$) i różnica między modelami dawki po zakończeniu leczenia.

In [ ]:
after = t >= end_of_treatment
rows = []
for name in plans:
    M = tumor_mass(runs[(name, "pulse")], x)
    MJ = tumor_mass(runs[(name, "jump")], x)
    m_min, t_min = min_mass(t, M)
    left, right = front_positions(x, runs[(name, "pulse")][-1])
    rows.append({
        "plan": name,
        "M_min / M(0)": m_min / M[0],
        "t_min": t_min,
        "czas do odrostu": time_to_regrowth(t, M),
        "lewy brzeg, t = 10": left,
        "prawy brzeg, t = 10": right,
        "skok LQ: czas do odrostu": time_to_regrowth(t, MJ),
        "skok LQ a impuls po leczeniu (największa różnica względna M)": np.max(np.abs(MJ[after] - M[after]) / M[after]),
    })
table = pd.DataFrame(rows)
table.to_csv(TABLES / "zad2_02_scenariusze.csv", index=False)
table.style.format(precision=3)

## Masa guza w czasie

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for (name, p), c in zip(plans.items(), plt.cm.tab10.colors):
    ax.plot(t, tumor_mass(runs[(name, "pulse")], x) / tumor_mass(runs[(name, "pulse")][:1], x), color=c, lw=2, label=name)
    ax.plot(t, tumor_mass(runs[(name, "jump")], x) / tumor_mass(runs[(name, "jump")][:1], x), color=c, lw=1, ls="--")
ax.axvspan(schedule.times[0] - 0.1, schedule.times[-1] + 0.1, color="0.92", lw=0)
ax.plot([], [], "k-", lw=2, label="impuls (model bazowy)")
ax.plot([], [], "k--", lw=1, label="skok LQ")
ax.set(xlabel="czas t", ylabel="M(t) / M(0)", title="Masa guza dla czterech planów (szare: okres frakcji)", ylim=(0, None))
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_02_masa.png", dpi=150)

## Skąd bierze się odrost: pole za wąskie i pole przesunięte

In [ ]:
snap = {k: int(np.argmin(np.abs(t - v))) for k, v in {"t = 0": 0.0, "koniec frakcji, t = 5": 5.0, "t = 10": 10.0}.items()}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, name in zip(axes, ["pole za wąskie", "pole przesunięte"]):
    u = runs[(name, "pulse")]
    ax.fill_between(x, 0, beam(x, plans[name]), color="tab:orange", alpha=0.15, label="pole wiązki R(x)")
    for lab, k in snap.items():
        ax.plot(x, u[k], lw=2, label=lab)
    ax.set(xlabel="x", title=name, xlim=(-30, 30))
axes[0].set_ylabel("gęstość u")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "zad2_02_profile.png", dpi=150)

## Wnioski

- **Pole pokrywa guz, duża dawka: zanik.** Masa spada praktycznie do zera i do $t = 10$ guz nie odrasta.
- **Pole pokrywa guz, mała dawka: częściowa odpowiedź.** Najmniejsza masa to ok. 20% $M(0)$, a guz wraca do $M(0)$ przy $t \approx 6{,}9$.
- **Pole za wąskie: najgorszy plan.** Środek guza ginie, ale oba brzegi rosną dalej, więc masa spada tylko do ok. 78% $M(0)$
  i przekracza $M(0)$ już w trakcie frakcji ($t \approx 2{,}8$). Guz odrasta symetrycznie z obu stron.
- **Pole przesunięte: odrost z jednej strony.** Najmniejsza masa to ok. 51% $M(0)$, odrost przy $t \approx 5{,}5$.
  W $t = 10$ lewy, nienapromieniony brzeg jest w $x \approx -23{,}2$ (tak samo jak przy polu za wąskim, bo wiązka go nie sięga),
  a prawy tylko w $x \approx 4{,}2$. Za prawym brzegiem pola rośnie też mały guzek z ogona, który wystawał poza pole.
- **Skok LQ i impuls dają prawie to samo.** Po zakończeniu frakcji masy różnią się o najwyżej 0,4%, a czas do odrostu o najwyżej 0,05
  poza polem za wąskim. Tam przy skoku masa tuż przed frakcją w $t = 2{,}6$ przekracza już $M(0)$ i dopiero frakcja ją cofa,
  więc odrost liczymy od 2,6, a przy impulsie od 2,75.
  Różnice widać tylko w trakcie frakcji, bo impuls zabija komórki przez czas swojej szerokości. Gładkie impulsy, wygodne dla PINN-ów,
  są więc dobrym przybliżeniem skoku z TumorTwin.

Wszystkie cztery wyniki zgadzają się z oczekiwaniami z prezentacji 23.10: poza polem człon dawki znika, a komórki, które tam zostały,
dzielą się i wracają w leczony obszar.